### LSTM for predicting anamolies in EGG timeseries data (epileptic seizures)

Dataset from https://www.kaggle.com/datasets/harunshimanto/epileptic-seizure-recognition?resource=download

The original dataset from the reference consists of 5 different folders, each with 100 files, with each file representing a single subject/person. Each file is a recording of brain activity for 23.6 seconds. The corresponding time-series is sampled into 4097 data points. Each data point is the value of the EEG recording at a different point in time. So we have total 500 individuals with each has 4097 data points for 23.5 seconds.

We divided and shuffled every 4097 data points into 23 chunks, each chunk contains 178 data points for 1 second, and each data point is the value of the EEG recording at a different point in time. So now we have 23 x 500 = 11500 pieces of information(row), each information contains 178 data points for 1 second(column), the last column represents the label y {1,2,3,4,5}.

The response variable is y in column 179, the Explanatory variables X1, X2, …, X178

y contains the category of the 178-dimensional input vector. Specifically y in {1, 2, 3, 4, 5}:

5 - eyes open, means when they were recording the EEG signal of the brain the patient had their eyes open

4 - eyes closed, means when they were recording the EEG signal the patient had their eyes closed

3 - Yes they identify where the region of the tumor was in the brain and recording the EEG activity from the healthy brain area

2 - They recorded the EEG from the area where the tumor was located

1 - Recording of seizure activity

All subjects falling in classes 2, 3, 4, and 5 are subjects who did not have epileptic seizure. Only subjects in class 1 have epileptic seizure. Our motivation for creating this version of the data was to simplify access to the data via the creation of a .csv version of it. Although there are 5 classes most authors have done binary classification, namely class 1 (Epileptic seizure) against the rest.

This Dataset collect from UCI Machine Learning Repository

### Setup

In [68]:
import numpy as np
import torch

global_seed = 1066
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

# Set random seeds for reproducibility
np.random.seed(global_seed)
torch.manual_seed(global_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(global_seed)

print(f"Using device: {device}")

%matplotlib inline

Using device: mps


### Data Processing and Handling

In [69]:
import pandas as pd

# 1 row => 1 second of recording at 178 HZ 
# to generalize to longer formats we will use sliding window technique 

df = pd.read_csv('data.csv')
# internally used as an ID, we don't need it
df = df.drop("Unnamed", axis=1)

print(df.info()) 
print(f"DataFrame overview: {df.head()}") 


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11500 entries, 0 to 11499
Columns: 179 entries, X1 to y
dtypes: int64(179)
memory usage: 15.7 MB
None
DataFrame overview:     X1   X2   X3   X4   X5   X6   X7   X8   X9  X10  ...  X170  X171  X172  \
0  135  190  229  223  192  125   55   -9  -33  -38  ...   -17   -15   -31   
1  386  382  356  331  320  315  307  272  244  232  ...   164   150   146   
2  -32  -39  -47  -37  -32  -36  -57  -73  -85  -94  ...    57    64    48   
3 -105 -101  -96  -92  -89  -95 -102 -100  -87  -79  ...   -82   -81   -80   
4   -9  -65  -98 -102  -78  -48  -16    0  -21  -59  ...     4     2   -12   

   X173  X174  X175  X176  X177  X178  y  
0   -77  -103  -127  -116   -83   -51  4  
1   152   157   156   154   143   129  1  
2    19   -12   -30   -35   -35   -36  5  
3   -77   -85   -77   -72   -69   -65  5  
4   -32   -41   -65   -83   -89   -73  5  

[5 rows x 179 columns]


### Splitting The Data

In [70]:
from sklearn.model_selection import train_test_split

# our dataset is perfectly balanced, each class has 2300 entries
print(f"DataFrame shape: {df.groupby('y').size()}")

# 70 15 15 split
train_df, temp = train_test_split(
    df, 
    test_size=0.3, 
    random_state=global_seed, 
    stratify=df['y']
)

validation_df, test_df = train_test_split(
    temp, 
    test_size=0.5, 
    random_state=global_seed, 
    stratify=temp['y']
)

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Training: {len(train_df)}")
print(f"Validation: {len(validation_df)}")
print(f"Test: {len(test_df)}")

DataFrame shape: y
1    2300
2    2300
3    2300
4    2300
5    2300
dtype: int64
Training: 8050
Validation: 1725
Test: 1725


### Data visualisation & Exploration

In [91]:
import plotly.graph_objects as go

class_titles = {
    1: "Active Seizure (1)",
    2: "Tumor Area (2)",
    3: "Healthy Area (3)",
    4: "Eyes Closed (4)",
    5: "Eyes Open (5)"
}

fig = go.Figure()

for item in class_titles:
    
    # one random sample from each class
    sample = train_df[train_df['y'] == item].sample(n=1)
    
    # convert to series
    voltages = sample.drop(columns=['y']).iloc[0]
    
    fig.add_trace(go.Scatter(
        x=voltages.index, 
        y=voltages.values, 
        mode='lines',
        name=class_titles.get(item, f"Class {item}") 
    ))

fig.update_layout(
    title="Overview of EEG Signals by Class",
    xaxis_title="Timestep (X1 - X178) - 1 second total",
    yaxis_title="Voltage",
    legend_title="EEG State",
    hovermode="x unified" 
)

fig.show()

In [ ]:
import plotly.express as px

voltages = train_df.drop(columns=['y'])

# standard deviation for each row
plot_df = pd.DataFrame({
    "std_dev":  voltages.std(axis=1),
    "class_name": train_df['y'].map(class_titles)
    })
print(f"Standard deviation of voltages (first 5 rows):\n{plot_df.head()}")

fig = px.box(plot_df,
             x="class_name", 
             y="std_dev", 
             labels={"class_name": "Patient State", 
                     "std_dev": "Standard Deviation"}, 
             title="Distribution of Standard Deviation by Patient State")
fig.show()

Standard deviation of voltages (first 5 rows):
     std_dev        class_name
0  47.142461     Eyes Open (5)
1  20.243939     Eyes Open (5)
2  25.083208  Healthy Area (3)
3  36.083253  Healthy Area (3)
4  74.310577   Eyes Closed (4)


we note that the standard deviation of the volage during and active seizure varies between 171 and 422 with a median of 278. This far dwarfs non-seizure voltage recorded. <br> We also note Tumor Area recordings have a significant amount of outlayers ranging from 130 to 541, overlapping with seizure activity.

Overall for **Binary Classification** the classes show good seperation. 

### LSTM Implemntation & Training

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

 
class DatasetInit(Dataset):
    def __init__(self, dataframe):
        self.labels = dataframe['y'].values
        self.features = dataframe.drop(columns=['y']).values

    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        x_tensor = torch.tensor(self.features[idx], dtype=torch.float32) # int better?
        x_tensor = x_tensor.unsqueeze(-1) # LSTM expects (batch_size, seq_len, input_size)
        y_tensor = torch.tensor(1.0 if self.labels[idx] == 1 else 0.0, dtype=torch.long)
        
        return x_tensor, y_tensor
    

train_dataset = DatasetInit(train_df)
vald_dataset = DatasetInit(dataframe=validation_df)
test_dataset = DatasetInit(dataframe=test_df)

batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
vald_loader = DataLoader(vald_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


# test  loaders if are working
features, labels = next(iter(train_loader))


print(f"image shape: {features.shape}")
print(f"batch shape {labels.shape}")
print(f"label type: {labels.dtype}")
print(f"first 5 labels: {labels[:5]}")  

image shape: torch.Size([64, 178, 1])
batch shape torch.Size([64])
label type: torch.int64
first 5 labels: tensor([1, 1, 0, 0, 1])
